<img src=https://courseware.decoded.com/images/decoded/logo-decoded.png align=left width=100px/>

# Time Series Analysis
---
<small>© COPYRIGHT 2026 DECODED LIMITED. ALL RIGHTS RESERVED. Web: https://decoded.com/ </small>

## Content

### Importing required libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import pmdarima as pm
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import adfuller, acf
from statsmodels.tsa.holtwinters import SimpleExpSmoothing
from sklearn.metrics import mean_squared_error
import math
import random
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# set plotting configurations
plt.style.use('bmh')
plt.rcParams['figure.figsize']=(10,5)
plt.rcParams['axes.autolimit_mode'] = 'round_numbers'
plt.rcParams['axes.xmargin'] = 0.
plt.rcParams['axes.ymargin'] = 0.
plt.rcParams.update({'font.size': 10})

In [ ]:
# round the column to be displayed by integer
pd.options.display.float_format = '{:,.0f}'.format

Loading in the data as csv and setting the frequency,

In [ ]:
ts = pd.read_csv('government_service_requests.csv')
ts.index=pd.date_range('2017-01-01', periods=len(ts), freq='W')
ts.tail()
# drop the date column


In [ ]:
# Drop the unnecessary columns
ts.drop(columns=['Unnamed: 0'], inplace=True)


### Training time series models

#### Decompose a time series with `seasonal_decompose()`

Decomposition involves breaking down a time series into its constituent parts: trend, seasonal and residual components. The `period` argument is automatically inferred, we can change this value to match the seasonality we believe the data exhibits.

Let's just decompose retail 

In [ ]:
# seasonal decomposition


Apply an additive decomposition by setting `model='a'`

Extract and plot the denoised additive model to see how well an additive composition fits the data,

#### Challenge: Is an additive or multiplicative decomposition best suited for this data?

- Apply a multiplicative decomposition by setting `model='m'`
- Extract the denoised multiplicative model by modifying `additive_model` and renaming to `multiplicative_model`
- Plot the denoised multiplicative model alongside the original time series
- Compare to the results of the additive decomposition, which best describes this data?

Note: If the variance of the residuals changes quite dramatically over time (known as heteroscedasticity) or shows a pattern, this suggests that the decomposition is not suitable for the data.

Extract and plot the denoised multiplicative model to see how well an multiplicative composition fits the data,

#### Challenge: Check for stationarity by eyeballing the time series graph

Let's plot the level of the series with a rolling window. Do you observe signs of non-stationarity?

#### Check for stationarity using `plot_acf()` from `statsmodels`



Run an Augmented Dickey-Fuller (ADF) test for stationarity using  `adfuller()`

#### Differencing a time series

We can stablise the mean of a time series through differencing `.diff()` and/or seasonal differencing `.diff(m)` to help make our time series stationary.

ARIMA models have a hyperparameter for differencing, it is important for us as analysts and programmers to determine what the hyperparameter `d` (differencing to reduce the trend) and `D` (differencing to reduce seasonality) should be.

#### Differencing with `.diff()` to reduce the trend

A way of removing/reducing trend in a time series is by differencing. Usually differencing once with `.diff()` is enough to make a time series stationary but we may need to apply second order differencing `.diff().diff()` or higher order differencing.

In [ ]:
# Take the differencing on your time series data to make it stationary

#### Challenge: Re-run `ACF` and `ADF` test, is our differenced time series stationary?

- Run the ADF test again to check for stationarity.
- The ADF test does not check for seasonality, so run an ACF plot again to check for seasonality.
- Jot down any results and observations.

Run the ADF test again to check for stationarity,

The p-value is now 0.0007 which is less than 0.05, and therefore we have sufficient evidence to reject the null hypothesis that the series has a unit root and it is NOT stationary.

### Time series forecasting - training the model

### How do you build a Time Series model in python?

Split the data with the first 80% of data points in the training set and the final 20% in the test set,

In [ ]:
threshold = int(len(ts['retail_closure']) * 0.8)
train = ts['retail_closure'][:threshold]
test = ts['retail_closure'][threshold:]

In [ ]:
train.tail()
test.tail()

#### Building a baseline time series model

First, it is good practice to establish a baseline model so that we have something to compare our model performances to.

For our baseline model we will use the Persistence Algorithm (the “naive” forecast).

This uses the previous value in the series to predict the expected outcome of the next value in the time series.

Generate the naive forecasts with `.shift()`

In [ ]:
# Build your base line model on the training data
ts['naive_pred'] = ts['retail_closure'].shift(1)

In [ ]:
ts[['naive_pred','retail_closure']].tail()
threshold = int(len(ts['retail_closure']) * 0.8)

ts['naive_pred'][threshold:]

Plot the naive forecasts for the test set,

In [ ]:
plt.plot(train, label="train", linestyle='--')
plt.plot(test, label="test", color="navy", linestyle='--')
# plot the naive predictions, which is the value of the previous time step, starting from the first test point
plt.plot(ts['naive_pred'][threshold:], label="naive predictions", marker = 'x', linestyle='--', color='red')
# plt.plot(ts_.naive_pred[threshold:], label="naive predictions", marker = 'x', linestyle='--', color='red')
plt.legend(loc='upper left');

To measure performance, we can use the RMSE evaluation metric by first calculating the MSE score using `mean_squared_error()` from `sklearn` and then taking the square root using `sqrt()` from `math` as follows,

In [ ]:
# calculate the test score using root mean squared error (RMSE)
test_score_naive = 

# test_score = math.sqrt(mean_squared_error(test.requests, ts_.naive_pred[threshold:]))
test_score_naive

#### Modelling with `AutoARIMA`

We can use `AutoARIMA` from `pmdarima` which will automatically determine the best hyperparameter values `(p,d,q)(P,D,Q)m`, where,

- p is the order (number of lags) incorporated into our model
- d is the degree of differencing (the number of times the data have had past values subtracted)
- q is the number of past error terms incorporated into our model
- P is the number of seasonal lags incorporated into our model
- D is the degree of seasonal differencing
- Q is the number of past seasonal error terms incorporated into our model
- m is the seasonal period

As analysts and programmers, it is good practice to determine and set `d` and `D` ourselves, as sometimes the model struggles to test and converge for these values. We should also provide a range of `p` and `q` to narrow the search space as follows,

In [ ]:
autoarima_model = pm.arima.AutoARIMA(start_p=0 ,max_p=1, d=0, start_q=0, max_q=1,
                                     start_P=0, max_P=1, D=1, start_Q=0, max_Q=1, m=13)

autoarima_model.fit(train)

We can look at a summary of the model generated by `AutoARIMA` using `.summary()`

Generate and plot the predictions made by the `AutoArima` model,

Use RMSE evaluation metric by using `mean_squared_error()` from `sklearn` and `sqrt()` from `math` to measure performance

In [ ]:

# 

# contains the evaluation metric for the test set for arima model   
test_score_autoarima = 

Compare the RMSE scores. what does it tell?

Therefore, we know our machine learning `AutoArima`	forecasting model is performing much better than the baseline model.

#### Challenge: Report key parameters of a time series model and intepret them

What hyperparameter values were outputted from `AutoArima` and are they the best choice based on our analysis?

- p=
- d=
- q=

- P=
- D=
- Q=

- m=

#### Create the the prediction interval of ARIMA

In [ ]:
# 

predict_period = test.shape[0]+12
start_date = test.index[0]
preds, confint = autoarima_model.predict(n_periods=predict_period, return_conf_int=True)
autoarima_preds = pd.Series(preds, index=pd.date_range(start_date, periods=predict_period, freq='W'))
autoarima_confint_lower = confint[:, 0]
autoarima_confint_upper = confint[:, 1]


In [ ]:
# Create the plot of the train, test, predictions and confidence intervals
plt.plot( _______________ , label="train", linestyle='--')
plt.plot( _______________, label="test", color="navy", linestyle='--')
plt.plot( _______________ , label="AutoArima predictions", marker = 'x', linestyle='--')
plt.fill_between(autoarima_preds.index, _______________ , _____________, alpha=.3, color='red')
plt.legend(loc='upper left');

<hr /> <div style='page-break-after: always; display: none'>\pagebreak</div>

### Tree-Based Time Series Forecasting

Tree-based models like Random Forest and XGBoost can be used for time series forecasting by creating lagged features from the time series data. This transforms the forecasting problem into a supervised learning regression task.

#### Feature Engineering for Time Series
To apply tree-based models, we create features such as:
- Lagged values (e.g., value at t-1, t-2, etc.)
- Rolling statistics (e.g., moving averages)
- Seasonal indicators
- Other exogeneous indicators
- A mixture of the above


#### Implementation in Python

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor , GradientBoostingRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
# Load the copied time series data
data = ts



#### We want to use lags as our features to build our machine learning models

In [ ]:
# Create lagged 1 and 2 as features 
# fill in the lag1


# create 13 week moving average as another feature

#### Split the feature and target time series

In [ ]:

# use spending as exogeneous variable to create the train test set



### Train test split

In [ ]:

# Train-test split (use last 26 weeks for testing)
threshold = int(len(data) * 0.8)
#
# How do we split the data into train and test sets for time series forecasting?
# We need to make sure that the training set contains only data from before the test set, to avoid data leakage.
X_train, X_test = X[:threshold], X[threshold:]

# how do we split the target variable?
y_train, y_test =                           

In [ ]:
# Train Random Forest model
rf_model = RandomForestRegressor(n_estimators=20, random_state=42)
rf_model.fit(X_train, y_train)


#### Calculate the performance using training data for Random forest

In [ ]:

rf_predictions_train = rf_model.predict(X_train)
rf_mse_train = mean_squared_error(y_train, rf_predictions_train)
rf_mae_train = mean_absolute_error(y_train, rf_predictions_train)
rf_rmse_train = np.sqrt(rf_mse_train)

print(f"Random Forest - Train MSE: {rf_mse_train:.2f}, Train MAE: {rf_mae_train:.2f}, Train RMSE: {rf_rmse_train:.2f}")


In [ ]:
# Let's see how the model performs on the test set

rf_predictions_test = 

#### Evaluate the performance of Gradient Boost using train data

**Task**

* Make predictions on the time series using test data
* Create the evaluation metrics 
* Print the result

In [ ]:
# Instantiate XGBoost model
gradient_model = GradientBoostingRegressor(n_estimators=20, random_state=42)
# train the model 

In [ ]:
# Create the evaluation metrics for the train set



* What do we observe from the RMSE here ? Better or worse than ARIMA and seasonal?

### Evaluate the performance using test data

In [ ]:
# 
# Make predictions
gradient_model_predictions = 


# Calculate metrics
gradient_model_mse = 
gradient_model_mae = 
gradient_model_rmse = 

print(f"Gradient Boosting - MSE: {gradient_model_mse:.2f}, MAE: {gradient_model_mae:.2f}, RMSE: {gradient_model_rmse:.2f}")

#### Visualise the results

In [ ]:
# Plot the results
plt.figure(figsize=(12, 6))
plt.plot(y_test.index, y_test.values, label='Actual', marker='o')
plt.plot(y_test.index, rf_predictions_test, label='Random Forest', marker='s')
plt.plot(y_test.index, gradient_model_predictions, label='Gradient Boosting', marker='^')
plt.title('Time Series Forecasting: Random Forest vs Gradient Boosting')
plt.xlabel('Date')
plt.ylabel('Passengers')
plt.legend()
plt.show()


### What do you see here? which one is better?

In [ ]:
### Write your comments on the performance of the two models and which one you would choose for this time series forecasting problem.

### Create a table to show the performance of all models

In [ ]:
# Put a table comparing the performance of all the models we have trained so far, including the naive forecast, SES, AutoARIMA, SARIMAX, Random Forest, and XGBoost. The table should include the RMSE for each model.
performance_data = {
    'Model': ['Naive Forecast', 'AutoARIMA', 'Random Forest', 'Gradient Boosting'],
    'RMSE': [test_score_naive, test_score_autoarima,  rf_rmse_test, gradient_model_rmse]
}
performance_df = pd.DataFrame(performance_data)
print(performance_df)

#### Performance Comparison
Tree-based models offer several advantages for time series forecasting:
- **Handling Nonlinearity**: They can capture complex nonlinear patterns in the data.
- **Feature Interactions**: Automatically model interactions between lagged features.
- **Robustness**: Less sensitive to outliers compared to linear models.
- **Interpretability**: Feature importance can provide insights into which lags are most predictive.

However, they may require more computational resources and careful hyperparameter tuning. In this example, we compared Random Forest and XGBoost on the air passengers dataset. The choice between them depends on the specific dataset and computational constraints.

### Review: Time Series Analysis

#### What did we learn?

* Describe a time series as a composition between trends, seasonality and residuals
* Describe the core terminology of time series analysis e.g. lags, stationarity, decomposition etc.
* Perform autocorrelation analysis
* Build and train a time series model
* Compare and contrast various approaches to time series analysis

#### What should I learn next?

#### Practice: Time Series
In the next workshop you will gain more experience with time series decomposition, understanding stationarity, pre-processing time series data for building forecasting models, and building and evaluating an ARIMA forecasting model.


#### Learning Objectives
- Choose the best decomposition model for a time series by comparing plots and residuals
- Implement and interpret ACF plots of a time series
- Identify stationarity and test for it using the ADF test, together with ACF plot
- Make a time series stationary through differencing
- Perform the steps to fit an ARIMA forecasting model to time series data
- Evaluate the performance of an ARIMA forecasting model

## Appendix

### What packages does this chapter use?

```python
import pandas as pd
from matplotlib import pyplot as plt
import pmdarima as pm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import adfuller, acf
from statsmodels.tsa.holtwinters import SimpleExpSmoothing
from sklearn.metrics import mean_squared_error
import math
import random
import warnings
warnings.filterwarnings("ignore")
```

---

# END